In [0]:
from pyspark.sql import functions as F

bronze_users = spark.table("fraud_project.bronze.users_data")
bronze_users.printSchema()

root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)



In [0]:
bronze_cards = spark.table("fraud_project.bronze.cards_data")
bronze_cards.printSchema()
display(bronze_cards.limit(5))

root
 |-- id: long (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: string (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: string (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: string (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: integer (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,YES,2,$33900,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,YES,1,$11600,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,YES,1,$19948,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,YES,2,$16400,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,YES,2,$19439,01/1997,2007,No


In [0]:
bronze_txn = spark.table("fraud_project.bronze.transactions_data")
bronze_txn.printSchema()
display(bronze_txn.limit(5))

root
 |-- id: long (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors: string (nullable = true)



id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01T00:01:00.000Z,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475328,2010-01-01T00:02:00.000Z,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,null
7475329,2010-01-01T00:02:00.000Z,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,null
7475331,2010-01-01T00:05:00.000Z,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,null
7475332,2010-01-01T00:06:00.000Z,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,null


In [0]:
from pyspark.sql import functions as F

bronze_users = spark.table("fraud_project.bronze.users_data")

silver_users = (bronze_users
    .withColumn("per_capita_income", F.regexp_replace(F.col("per_capita_income"), "[$,]", "").cast("double"))
    .withColumn("yearly_income", F.regexp_replace(F.col("yearly_income"), "[$,]", "").cast("double"))
    .withColumn("total_debt", F.regexp_replace(F.col("total_debt"), "[$,]", "").cast("double"))
    .dropDuplicates(["id"])
    .withColumnRenamed("id", "client_id")
)

print("Row count:", silver_users.count())
display(silver_users.limit(10))

Row count: 2000


client_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,20599.0,41997.0,0.0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,25258.0,51500.0,102286.0,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,26790.0,54623.0,114711.0,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,26273.0,42509.0,2895.0,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,18730.0,38190.0,81262.0,810,1


In [0]:
silver_users.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fraud_project.silver.users")
spark.sql("SELECT COUNT(*) FROM fraud_project.silver.users").show()

+--------+
|COUNT(*)|
+--------+
|    2000|
+--------+



In [0]:
bronze_cards = spark.table("fraud_project.bronze.cards_data")

silver_cards = (bronze_cards
    .withColumn("credit_limit", F.regexp_replace(F.col("credit_limit"), "[$,]", "").cast("double"))
    .withColumn("expires", F.to_date("expires", "MM/yyyy"))
    .withColumn("acct_open_date", F.to_date("acct_open_date", "MM/yyyy"))
    .withColumn("has_chip", F.when(F.col("has_chip") == "YES", True).otherwise(False))
    .withColumn("card_on_dark_web", F.when(F.col("card_on_dark_web") == "Yes", True).otherwise(False))
    .dropDuplicates(["id"])
    .withColumnRenamed("id", "card_id")
)

print("Row count:", silver_cards.count())
display(silver_cards.limit(10))

Row count: 6146


card_id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,2024-04-01,866,true,2,33900.0,1991-01-01,2014,false
1,550,Mastercard,Credit,5278231764792292,2024-06-01,396,true,1,11600.0,1994-01-01,2013,false
2,556,Mastercard,Debit,5889825928297675,2021-09-01,422,true,1,19948.0,1995-01-01,2011,false
3,1937,Visa,Credit,4289888672554714,2020-04-01,736,true,2,16400.0,1995-01-01,2015,false
4,1981,Mastercard,Debit,5433366978583845,2024-03-01,530,true,2,19439.0,1997-01-01,2007,false
5,619,Visa,Debit,4657824650820465,2024-04-01,245,true,2,21883.0,1997-01-01,2012,false
6,1046,Amex,Credit,394584924614148,1999-02-01,302,true,2,9400.0,1998-01-01,2011,false
7,511,Mastercard,Debit,5585238056278288,2005-03-01,749,true,1,9664.0,1998-01-01,2011,false
8,1107,Mastercard,Credit,5462760953855576,2021-09-01,665,false,2,10300.0,1998-01-01,2006,false
9,1046,Amex,Credit,357982644067712,2020-09-01,72,true,1,13000.0,1999-01-01,2005,false


In [0]:
silver_cards.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fraud_project.silver.cards")
spark.sql("SELECT COUNT(*) FROM fraud_project.silver.cards").show()

+--------+
|COUNT(*)|
+--------+
|    6146|
+--------+



In [0]:
bronze_cards.select("has_chip").distinct().show()
bronze_cards.select("card_on_dark_web").distinct().show()

+--------+
|has_chip|
+--------+
|     YES|
|      NO|
+--------+

+----------------+
|card_on_dark_web|
+----------------+
|              No|
+----------------+



In [0]:
bronze_txn = spark.table("fraud_project.bronze.transactions_data")
mcc_df = spark.table("fraud_project.bronze.mcc_codes")
fraud_df = spark.table("fraud_project.bronze.train_fraud_labels")

# Clean mcc_df types (mcc_code came through as string keys from the JSON reshape)
mcc_df = mcc_df.withColumn("mcc_code", F.col("mcc_code").cast("integer"))

# Clean fraud_df (transaction_id came through as string keys)
fraud_df = (fraud_df
    .withColumn("transaction_id", F.col("transaction_id").cast("long"))
    .withColumn("is_fraud", F.when(F.col("is_fraud_label") == "Yes", True).otherwise(False))
    .select("transaction_id", "is_fraud")
)

silver_txn = (bronze_txn
    .withColumn("amount", F.regexp_replace(F.col("amount"), "[$,]", "").cast("double"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn("hour_of_day", F.hour("date"))
    .withColumn("is_weekend", F.dayofweek("date").isin([1, 7]))
    .withColumn(
        "time_bucket",
        F.when((F.hour("date") >= 5) & (F.hour("date") <= 11), "Morning")
         .when((F.hour("date") >= 12) & (F.hour("date") <= 16), "Afternoon")
         .when((F.hour("date") >= 17) & (F.hour("date") <= 20), "Evening")
         .otherwise("Night")
    )
    .join(mcc_df, F.col("mcc") == F.col("mcc_code"), how="left")
    .join(fraud_df, F.col("id") == F.col("transaction_id"), how="left")
    .withColumn("is_fraud", F.coalesce(F.col("is_fraud"), F.lit(False)))  # documented assumption: unlabeled = non-fraud
    .drop("mcc_code", "transaction_id")
    .withColumnRenamed("id", "transaction_id")
)

print("Row count:", silver_txn.count())
display(silver_txn.limit(10))

Row count: 13305915


transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,day_of_week,hour_of_day,is_weekend,time_bucket,mcc_description,is_fraud
7475498,2010-01-01T04:27:00.000Z,1736,113,55.79,Swipe Transaction,24823,Brooklyn,NY,11211.0,7538,null,Friday,4,false,Night,Automotive Service Shops,false
7475758,2010-01-01T06:55:00.000Z,428,3842,78.66,Swipe Transaction,90865,Lebanon,NH,3766.0,5311,null,Friday,6,false,Morning,Department Stores,false
7475895,2010-01-01T07:25:00.000Z,1435,4294,70.92,Swipe Transaction,27601,Alpharetta,GA,30004.0,7538,null,Friday,7,false,Morning,Automotive Service Shops,false
7475936,2010-01-01T07:37:00.000Z,419,3854,2.28,Swipe Transaction,55730,Murphys,CA,95247.0,5812,null,Friday,7,false,Morning,Eating Places and Restaurants,false
7476137,2010-01-01T08:27:00.000Z,1453,1117,-145.0,Swipe Transaction,16790,Spring Valley,NY,10977.0,3389,null,Friday,8,false,Morning,Non-Precious Metal Services,false
7476200,2010-01-01T08:43:00.000Z,94,2890,28.22,Online Transaction,39021,ONLINE,null,null,4784,null,Friday,8,false,Morning,Tolls and Bridge Fees,false
7477082,2010-01-01T11:56:00.000Z,1169,5763,53.64,Online Transaction,41122,ONLINE,null,null,4784,null,Friday,11,false,Morning,Tolls and Bridge Fees,false
7477530,2010-01-01T13:18:00.000Z,1696,2408,89.0,Swipe Transaction,61195,Merritt Island,FL,32952.0,5541,null,Friday,13,false,Afternoon,Service Stations,false
7478036,2010-01-01T15:22:00.000Z,1659,185,33.0,Online Transaction,16798,ONLINE,null,null,4121,null,Friday,15,false,Afternoon,Taxicabs and Limousines,false
7478324,2010-01-01T16:35:00.000Z,1897,4948,23.13,Online Transaction,39021,ONLINE,null,null,4784,null,Friday,16,false,Afternoon,Tolls and Bridge Fees,false


In [0]:
(silver_txn.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("day_of_week")  # optional, speeds up day-of-week gold queries; skip if it errors
    .saveAsTable("fraud_project.silver.transactions")
)

spark.sql("SELECT COUNT(*) FROM fraud_project.silver.transactions").show()

+--------+
|COUNT(*)|
+--------+
|13305915|
+--------+



In [0]:
(silver_txn.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_project.silver.transactions")
)

In [0]:
# Fraud rate overall
spark.sql("""
    SELECT is_fraud, COUNT(*) as cnt, ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 3) as pct
    FROM fraud_project.silver.transactions
    GROUP BY is_fraud
""").show()

# Confirm mcc join worked
spark.sql("SELECT COUNT(*) FROM fraud_project.silver.transactions WHERE mcc_description IS NULL").show()

# Confirm date parsing worked
spark.sql("SELECT MIN(date), MAX(date) FROM fraud_project.silver.transactions").show()

+--------+--------+------+
|is_fraud|     cnt|   pct|
+--------+--------+------+
|    true|   13332| 0.100|
|   false|13292583|99.900|
+--------+--------+------+

+--------+
|COUNT(*)|
+--------+
|       0|
+--------+

+-------------------+-------------------+
|          MIN(date)|          MAX(date)|
+-------------------+-------------------+
|2010-01-01 00:01:00|2019-10-31 23:59:00|
+-------------------+-------------------+



In [0]:
spark.sql("SELECT COUNT(*) FROM fraud_project.silver.users").show()
display(spark.sql("SELECT * FROM fraud_project.silver.users LIMIT 5"))

spark.sql("SELECT COUNT(*) FROM fraud_project.silver.cards").show()
display(spark.sql("SELECT * FROM fraud_project.silver.cards LIMIT 5"))

+--------+
|COUNT(*)|
+--------+
|    2000|
+--------+



client_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,18730.0,38190.0,81262.0,810,1
640,29,63,1990,9,Female,8677 Littlewood Lane,40.42,-104.74,22427.0,45727.0,94016.0,629,1
633,36,69,1983,10,Female,5506 Fifth Boulevard,33.88,-118.27,24611.0,50179.0,110515.0,743,1
668,87,57,1932,10,Female,8569 Wessex Boulevard,33.52,-86.79,13263.0,16342.0,1758.0,747,5
1943,19,65,2000,9,Female,1134 Valley Drive,41.71,-72.83,46762.0,95348.0,71972.0,706,3


+--------+
|COUNT(*)|
+--------+
|    6146|
+--------+



card_id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
28,1852,Visa,Debit,4013876625955765,2023-10-01,543,true,2,36425.0,2002-01-01,2004,false
29,316,Visa,Credit,4934996934968532,2021-04-01,547,true,3,1200.0,2002-01-01,2011,false
30,261,Visa,Credit,4572295668772647,2022-04-01,795,true,2,14200.0,2002-01-01,2006,false
33,1944,Visa,Credit,4803534694306012,2021-09-01,383,true,1,11100.0,2002-01-01,2009,false
93,311,Discover,Credit,6129847719830710,2024-08-01,972,true,1,1800.0,2006-01-01,2007,false


In [0]:
spark.sql("SELECT has_chip, COUNT(*) FROM fraud_project.silver.cards GROUP BY has_chip").show()
spark.sql("SELECT card_on_dark_web, COUNT(*) FROM fraud_project.silver.cards GROUP BY card_on_dark_web").show()

+--------+--------+
|has_chip|COUNT(*)|
+--------+--------+
|    true|    5500|
|   false|     646|
+--------+--------+

+----------------+--------+
|card_on_dark_web|COUNT(*)|
+----------------+--------+
|           false|    6146|
+----------------+--------+



In [0]:
spark.table("fraud_project.bronze.cards_data").select("card_on_dark_web").distinct().show()

+----------------+
|card_on_dark_web|
+----------------+
|              No|
+----------------+

